# Train the L2 fraud-detection model on Colab

Runs two training jobs against a GPU runtime and hands you back checkpoints in the same format `packages/models/train_baseline.py` and `packages/fl/server.py` produce locally, so they drop straight into `packages/models/checkpoints/` when you're done:

1. **Centralised baseline** on the real Elliptic dataset (165 anonymised features) — `baseline.pt`.
2. **Federated run** (FedProx, 3 simulated non-IID clients) on the L2 simulator's 4-feature schema (`value_in`, `value_out`, `degree_in`, `degree_out`) — `federated_fedprox.pt`. This one is what the app's *Test a Transaction* tab can score, since that feature only accepts the 4-feature simulator schema (Elliptic's features are anonymised, so there's no meaningful way to hand-build a transaction against a baseline checkpoint).

**Before you start:** Runtime → Change runtime type → GPU (T4 is enough).

**Security note:** do **not** pass `--on-chain` to either script in this notebook. That path needs `DEPLOYER_PRIVATE_KEY` / client keys from your local `.env` / `wallets.local.json`, and Colab notebooks (and their outputs) are easy to accidentally leave in Drive or share with secrets embedded. Do on-chain commitment runs locally only, straight from your machine.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the repository onto this runtime

The repo is public at `github.com/nasredeenabdulhaleem/l2-fraud-fl`, so a plain clone works with no auth needed. This cell is safe to re-run: it clones only if the directory isn't already there, then pulls, so a runtime that already has the repo picks up new commits in place instead of failing with *destination path already exists*.

Either way you get whatever is on `main` right now — if you've made local commits since your last push, push them first so Colab sees them (uncommitted local changes never show up here regardless). `--ff-only` means a pull that can't fast-forward stops loudly rather than dropping you into a merge inside a notebook.

If you pull code changes after having already run the cells below, restart the session (Runtime → Restart session) before re-running: the `pip install -e .` install is editable, so the FL subprocesses launched further down pick up new code on their own, but modules this kernel has already imported stay as they were.

In [ ]:
import os

# Clone-or-pull, so re-running this in a runtime that already has the repo
# updates it instead of erroring out on an existing directory.
if not os.path.isdir("/content/l2-fraud-fl"):
    !git clone https://github.com/nasredeenabdulhaleem/l2-fraud-fl.git /content/l2-fraud-fl

%cd /content/l2-fraud-fl
!git pull --ff-only

## 2. Install dependencies

Colab's runtime already ships a CUDA-matched `torch` plus `numpy`, `pandas`, `scikit-learn`, `networkx` and `requests`. `requirements.txt` pins every one of those to a version that predates the Python this runtime uses (3.13 at the time of writing), so pip finds no matching wheel, falls back to building from source, and dies partway down the file — taking the rest of the file with it, which is why an install driven from `requirements.txt` can leave you with no `torch_geometric` at all. Those pins are for a local 3.10/3.11 environment. Here we install only what this runtime is genuinely missing, unpinned, so pip resolves the current release of each against whatever Colab is shipping that week rather than dragging the preinstalled stack backwards.

The one exception is the cap on `flwr`, which is a compatibility bound rather than a version preference: `packages/fl/server.py` and `packages/fl/client.py` call `fl.server.start_server` / `fl.client.start_client`, and Flower deleted that API in 1.27 (it survives, deprecated, through 1.26.x). Without the cap the federated section below fails outright — lifting it means migrating the FL code to Flower's `ServerApp`/`ClientApp` API first.

`web3` is here even though nothing on-chain runs in this notebook: `packages/fl/server.py` imports `packages.fl.telemetry`, which imports `packages.chain.aggregator_client`, whose `from web3 import Web3` executes at module load time regardless.

Installs go through `{sys.executable} -m pip` rather than bare `!pip`, so they land in the exact interpreter this notebook (and the background FL processes below, which launch via `sys.executable` too) actually runs — a bare `!pip`/`!python` pair can silently resolve to a different interpreter in some Colab images, which is the most common reason `import torch_geometric` or `import flwr` fails right after an install cell that looked like it succeeded.

pip will likely print dependency-resolver complaints about unrelated preinstalled Colab packages; those are noise. What matters is the version block the next cell prints. If an import there fails, do Runtime → Restart session and re-run this cell — a package that was upgraded in place while already imported needs a fresh interpreter.

In [ ]:
import sys

# Only the packages this runtime lacks, at whatever pip resolves as current --
# read the note above before adding to this list. The flwr cap is an API
# compatibility bound, not a pin: start_server/start_client are gone in 1.27.
!{sys.executable} -m pip install torch-geometric "flwr<1.27" web3
!{sys.executable} -m pip install -e .

# Verify right here, loudly, instead of finding out three cells and a 690MB
# download later -- {sys.executable} is also what the background FL
# processes below launch with, so this is the exact interpreter that matters.
import torch
import torch_geometric
import flwr
import web3

print("python", sys.version.split()[0])
print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())
print("torch_geometric", torch_geometric.__version__)
print("flwr", flwr.__version__)
print("web3", web3.__version__)

## 3. Centralised baseline on Elliptic

`EllipticLoader` downloads the raw CSVs via PyTorch Geometric on first use if they aren't already under `data/elliptic/raw/` — the feature file is ~690MB uncompressed, so the first run takes a few minutes.

In [ ]:
!{sys.executable} -m packages.models.train_baseline --source elliptic --epochs 30 --out packages/models/checkpoints/baseline.pt

## 4. Federated run (FedProx, 3 clients)

`packages/fl/server.py` and `packages/fl/client.py` talk over real sockets (`--server 127.0.0.1:8080`), not Flower's simulation API — there's no `start_simulation` entry point in this codebase. That still works fine in one Colab runtime: launch the server and three clients as background processes on localhost, exactly like running them in four terminals locally. `--no-telemetry` skips posting to the dashboard backend (nothing to post to from here); `--on-chain` is deliberately omitted per the note at the top.

In [ ]:
import subprocess
import sys
import time
from pathlib import Path

LOG_DIR = Path("/content/fl_logs")
LOG_DIR.mkdir(exist_ok=True)
NUM_CLIENTS = 3

def spawn(name, args):
    log = open(LOG_DIR / f"{name}.log", "w")
    return subprocess.Popen([sys.executable, "-m", *args], stdout=log, stderr=subprocess.STDOUT)

server_proc = spawn("server", [
    "packages.fl.server", "--strategy", "fedprox", "--rounds", "10",
    "--min-clients", str(NUM_CLIENTS), "--no-telemetry",
])
time.sleep(5)  # give the server time to bind before clients try to connect

client_procs = [
    spawn(f"client_{i}", [
        "packages.fl.client", "--client-id", str(i), "--num-clients", str(NUM_CLIENTS),
        "--server", "127.0.0.1:8080",
    ])
    for i in range(NUM_CLIENTS)
]

print(f"server pid={server_proc.pid}, client pids={[p.pid for p in client_procs]}")
print(f"logs under {LOG_DIR}")

In [ ]:
TIMEOUT_S = 20 * 60
waited = 0
while any(p.poll() is None for p in client_procs) and waited < TIMEOUT_S:
    time.sleep(10)
    waited += 10

if waited >= TIMEOUT_S:
    print("timed out waiting for clients -- check the logs below before continuing")
else:
    print(f"all clients finished after ~{waited}s")

# The server saves its checkpoint right after the last round finalises, then exits.
server_proc.wait(timeout=60)
print("server exit code:", server_proc.returncode)

for log_file in sorted(LOG_DIR.glob("*.log")):
    print(f"\n----- {log_file.name} (last 15 lines) -----")
    print("\n".join(log_file.read_text().splitlines()[-15:]))

## 5. Download the checkpoints

Save these back into your local `packages/models/checkpoints/`. Any checkpoint with `in_dim == 4` (the federated one) shows up automatically in the app's *Test a Transaction* model picker the next time the backend starts; the Elliptic baseline (`in_dim == 165`) is listed too but marked not scorable there, since its features aren't something you can hand-type.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/checkpoints", "zip", "packages/models/checkpoints")
files.download(archive)